For a target month, each target day D uses the GFS 00Z run from D−1,
with forecast leads 21, 24, ..., 48 h.

Therefore gfs_2016_01.nc begins with the 2015-12-31 00Z run.

In [ ]:
from pathlib import Path
import xarray as xr
import pandas as pd

from energy_forecasting.weather import (
    download_gfs_run,
    download_gfs_month,
    download_and_save_month,
    )

In [2]:
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

weather_dir = project_root / "data" / "raw" / "weather"

In [2]:
weather_test = download_gfs_run("20191114")
weather_test

<xarray.Dataset> Size: 93kB
Dimensions:                        (datetime_utc: 10, latitude: 39,
                                    longitude: 59)
Coordinates:
  * datetime_utc                   (datetime_utc) datetime64[ns] 80B 2019-11-...
    forecast_hour                  (datetime_utc) int64 80B 21 24 27 ... 45 48
  * latitude                       (latitude) float32 156B 44.5 44.25 ... 35.0
  * longitude                      (longitude) float32 236B -10.0 -9.75 ... 4.5
    reftime                        datetime64[ns] 8B 2019-11-14
Data variables:
    temperature_K                  (datetime_utc, latitude, longitude) float32 92kB ...
    LatLon_721X1440-0p13S-180p00E  (datetime_utc) int32 40B -2147483647 ... -...
Attributes: (12/13)
    Originating_or_generating_Center:                                        ...
    Originating_or_generating_Subcenter:                                     ...
    GRIB_table_version:                                                      ...
    Type_of_generating_process:                                              ...
    Analysis_or_forecast_generating_process_identifier_defined_by_originating...
    Conventions:                                                             ...
    ...                                                                                ...
    featureType:                                                             ...
    History:                                                                 ...
    geospatial_lat_min:                                                      ...
    geospatial_lat_max:                                                      ...
    geospatial_lon_min:                                                      ...
    geospatial_lon_max:                                                      ...

In [3]:
new_weather_test = download_gfs_month("2022", "12")

Retry run=20221201, fh=36, attempt=1/3
Retry run=20221201, fh=36, attempt=2/3
Retry run=20221201, fh=36, attempt=3/3
Retry run=20221215, fh=30, attempt=1/3
Retry run=20221222, fh=48, attempt=1/3


In [11]:
new_weather_test

<xarray.Dataset> Size: 3MB
Dimensions:         (run_time_utc: 31, forecast_hour: 10, latitude: 39,
                     longitude: 59)
Coordinates:
  * run_time_utc    (run_time_utc) datetime64[ns] 248B 2022-11-30 ... 2022-12-30
  * forecast_hour   (forecast_hour) int64 80B 21 24 27 30 33 36 39 42 45 48
  * latitude        (latitude) float32 156B 44.5 44.25 44.0 ... 35.5 35.25 35.0
  * longitude       (longitude) float32 236B -10.0 -9.75 -9.5 ... 4.0 4.25 4.5
Data variables:
    temperature_K   (run_time_utc, forecast_hour, latitude, longitude) float32 3MB ...
    lead_available  (run_time_utc, forecast_hour) bool 310B True True ... True
Attributes: (12/13)
    Originating_or_generating_Center:                                        ...
    Originating_or_generating_Subcenter:                                     ...
    GRIB_table_version:                                                      ...
    Type_of_generating_process:                                              ...
    Analysis_or_forecast_generating_process_identifier_defined_by_originating...
    Conventions:                                                             ...
    ...                                                                                ...
    featureType:                                                             ...
    History:                                                                 ...
    geospatial_lat_min:                                                      ...
    geospatial_lat_max:                                                      ...
    geospatial_lon_min:                                                      ...
    geospatial_lon_max:                                                      ...

In [ ]:
for year in range(2016, 2027):
    last_month = 8 if year == 2026 else 12

    for month in range(1, last_month + 1):
        download_and_save_month(
            year=year,
            month=month,
            output_dir=weather_dir,
        )

In [8]:
weather_files = sorted(
    weather_dir.glob("gfs_????_??.nc")
)

assert len(weather_files) == 128

print("Monthly files:", len(weather_files))

Monthly files: 128


In [7]:
total_runs = 0
missing_leads = []

for path in weather_files:
    with xr.open_dataset(path) as ds:
        total_runs += ds.sizes["run_time_utc"]

        assert ds.sizes["forecast_hour"] == 10
        assert ds.sizes["latitude"] == 39
        assert ds.sizes["longitude"] == 59

        assert ds["forecast_hour"].values.tolist() == list(
            range(21, 49, 3)
        )

        lead_available = ~(
             ds["temperature_K"].isnull()
             .all(dim=("latitude", "longitude"))
             )
        
        missing = ~lead_available

        if missing.any():
            positions = missing.where(missing, drop=True)

            for run_time in positions["run_time_utc"].values:
                for forecast_hour in positions["forecast_hour"].values:
                    if bool(
                        missing.sel(
                            run_time_utc=run_time,
                            forecast_hour=forecast_hour,
                        )
                    ):
                        missing_leads.append(
                            (run_time, int(forecast_hour))
                        )

print("Total runs:", total_runs)
print("Missing leads:", len(missing_leads))
print(missing_leads)

Total runs: 3896
Missing leads: 5
[(np.datetime64('2016-02-20T00:00:00.000000000'), 39), (np.datetime64('2016-02-22T00:00:00.000000000'), 27), (np.datetime64('2016-09-26T00:00:00.000000000'), 36), (np.datetime64('2022-12-01T00:00:00.000000000'), 36), (np.datetime64('2024-09-15T00:00:00.000000000'), 39)]


In [11]:
run_times = []

for path in weather_files:
    with xr.open_dataset(path) as ds:
        run_times.extend(
            pd.to_datetime(ds["run_time_utc"].values)
        )

run_times = pd.DatetimeIndex(run_times).sort_values()

assert len(run_times) == 3896
assert run_times.is_unique
assert (
    run_times.to_series().diff().dropna()
    == pd.Timedelta(days=1)
).all()

print("First run:", run_times.min())
print("Last run:", run_times.max())

First run: 2015-12-31 00:00:00
Last run: 2026-08-30 00:00:00
